# 11b — Privileged Text-Teacher KD on ProcedureVRL Hidden Features

This notebook is an improved text-assisted experiment after notebook 11.

Notebook 11 used a fixed text-prototype teacher and produced a negative/neutral result. This notebook uses a stronger **privileged text teacher**:

```text
Teacher training input:  ProcedureVRL hidden video features + ground-truth action text embeddings per timestep
Student training input:  ProcedureVRL hidden video features only
Student inference input: ProcedureVRL hidden video features only
```

The teacher is an oracle/privileged model and is **not** a deployable inference model. The final KD student remains video-only.

Main success criterion:

```text
KD student > CE-only student trained in this same notebook
```

Secondary criterion:

```text
KD student > previous notebook-10 hidden visual-only baseline
```

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import json, time, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Mounted at /content/drive
PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


In [2]:
# =====================
# Configuration
# =====================
DRIVE_ROOT = Path('/content/drive/MyDrive/mmf_tas_lab_data')

DATA_ROOT = (
    DRIVE_ROOT
    / 'text_assisted_tas'
    / 'breakfast'
    / 'procedurevrl_hidden'
    / 'runs'
    / 'procedurevrl_hidden_full_split1_split1_views16'
    / 'mstcn_format'
)

FEATURE_DIR = DATA_ROOT / 'features'
GT_DIR = DATA_ROOT / 'groundTruth'
SPLIT_DIR = DATA_ROOT / 'splits'
MAPPING_PATH = DATA_ROOT / 'mapping.txt'

TEXT_EMB_PATH = (
    DRIVE_ROOT
    / 'text_assisted_tas'
    / 'breakfast'
    / 'text_embeddings'
    / 'breakfast_clip_vitb16_text_embeddings.npy'
)
TEXT_META_PATH = (
    DRIVE_ROOT
    / 'text_assisted_tas'
    / 'breakfast'
    / 'text_embeddings'
    / 'breakfast_clip_vitb16_text_embedding_metadata.csv'
)

SPLIT_ID = 1

# Run smoke first if you want a quick integrity check.
# RUN_MODE = 'smoke'
RUN_MODE = 'full_split1'

RUN_CONFIGS = {
    'smoke': {
        'max_train_videos': 64,
        'max_test_videos': 32,
        'ce_epochs': 3,
        'teacher_epochs': 3,
        'student_epochs': 3,
        'eval_every': 1,
        'kd_lambdas': [0.05],
    },
    'full_split1': {
        'max_train_videos': None,
        'max_test_videos': None,
        'ce_epochs': 120,
        'teacher_epochs': 80,
        'student_epochs': 120,
        'eval_every': 5,
        # Small sweep because too much KD can hurt.
        'kd_lambdas': [0.01, 0.05, 0.10],
    },
}

cfg = RUN_CONFIGS[RUN_MODE]

OUT_ROOT = (
    DRIVE_ROOT
    / 'text_assisted_tas'
    / 'breakfast'
    / 'procedurevrl_hidden'
    / 'runs'
    / f'procedurevrl_hidden_privileged_text_teacher_kd_{RUN_MODE}_split{SPLIT_ID}'
)
RESULTS_DIR = OUT_ROOT / 'results'
MODEL_DIR = OUT_ROOT / 'models'
PRED_DIR = OUT_ROOT / 'predictions'
for p in [OUT_ROOT, RESULTS_DIR, MODEL_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SEED = 7
BATCH_SIZE = 64
NUM_WORKERS = 2

INPUT_DIM = 512
TEXT_DIM = 512
NUM_STAGES = 4
NUM_LAYERS = 10
NUM_F_MAPS = 64
DROPOUT = 0.5

LR = 5e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 5.0
SMOOTHING_WEIGHT = 0.15
KD_TEMPERATURE = 4.0

CE_EPOCHS = cfg['ce_epochs']
TEACHER_EPOCHS = cfg['teacher_epochs']
STUDENT_EPOCHS = cfg['student_epochs']
EVAL_EVERY = cfg['eval_every']
KD_LAMBDAS = cfg['kd_lambdas']

STANDARDIZE_FEATURES = True
NORMALIZE_TEXT_EMBEDDINGS = True

PREVIOUS_HIDDEN_BASELINE = {
    'experiment': 'ProcedureVRL hidden visual-only baseline',
    'best_epoch': 55,
    'acc': 58.04,
    'edit': 55.57,
    'f1@10': 58.58,
    'f1@25': 57.73,
    'f1@50': 45.76,
    'note': 'Notebook 10, 120 epochs, best checkpoint at epoch 55.',
}

print('RUN_MODE:', RUN_MODE)
print('OUT_ROOT:', OUT_ROOT)
print('KD_LAMBDAS:', KD_LAMBDAS)
for p in [DATA_ROOT, FEATURE_DIR, GT_DIR, SPLIT_DIR, MAPPING_PATH, TEXT_EMB_PATH]:
    print(p, '->', p.exists())
    assert p.exists(), f'Missing path: {p}'

RUN_MODE: full_split1
OUT_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_privileged_text_teacher_kd_full_split1_split1
KD_LAMBDAS: [0.01, 0.05, 0.1]
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_full_split1_split1_views16/mstcn_format -> True
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_full_split1_split1_views16/mstcn_format/features -> True
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_full_split1_split1_views16/mstcn_format/groundTruth -> True
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_full_split1_split1_views16/mstcn_format/splits -> True
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_h

In [3]:
# =====================
# Utilities and data loading
# =====================
def set_seed(seed=7):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(SEED)


def read_lines(path):
    return [x.strip() for x in Path(path).read_text().splitlines() if x.strip()]


def load_mapping(path):
    id_to_label = {}
    label_to_id = {}
    for line in read_lines(path):
        idx, lab = line.split()[:2]
        id_to_label[int(idx)] = lab
        label_to_id[lab] = int(idx)
    return id_to_label, label_to_id


def load_bundle(path):
    ids = []
    for line in read_lines(path):
        if line.endswith('.txt'):
            line = line[:-4]
        ids.append(line)
    return ids

id_to_label, label_to_id = load_mapping(MAPPING_PATH)
num_classes = len(id_to_label)

train_ids = load_bundle(SPLIT_DIR / f'train.split{SPLIT_ID}.bundle')
test_ids = load_bundle(SPLIT_DIR / f'test.split{SPLIT_ID}.bundle')
if cfg['max_train_videos'] is not None:
    train_ids = train_ids[:cfg['max_train_videos']]
if cfg['max_test_videos'] is not None:
    test_ids = test_ids[:cfg['max_test_videos']]

text_embeddings = np.load(TEXT_EMB_PATH).astype(np.float32)
assert text_embeddings.shape == (num_classes, TEXT_DIM), (text_embeddings.shape, num_classes, TEXT_DIM)

# If metadata contains clear label names, reorder. Otherwise assume mapping-id order.
if TEXT_META_PATH.exists():
    text_meta = pd.read_csv(TEXT_META_PATH)
    print('Text metadata columns:', list(text_meta.columns))
    display(text_meta.head())
else:
    text_meta = None

label_col = None
if text_meta is not None:
    for col in ['label', 'action', 'action_label', 'class_name', 'class_label', 'name', 'text']:
        if col in text_meta.columns:
            vals = set(str(x) for x in text_meta[col].tolist())
            overlap = len(vals.intersection(set(label_to_id.keys())))
            if overlap >= int(0.8 * num_classes):
                label_col = col
                break

if label_col is not None:
    print('Detected text label column:', label_col)
    row_by_label = {str(row[label_col]): i for i, row in text_meta.iterrows()}
    reordered = np.zeros_like(text_embeddings)
    for cid, label in id_to_label.items():
        assert label in row_by_label, label
        reordered[cid] = text_embeddings[row_by_label[label]]
    text_embeddings = reordered
else:
    print('Assuming text embeddings are already in mapping-id order.')

if NORMALIZE_TEXT_EMBEDDINGS:
    text_embeddings = text_embeddings / (np.linalg.norm(text_embeddings, axis=1, keepdims=True) + 1e-8)

print('num_classes:', num_classes)
print('train/test:', len(train_ids), len(test_ids))
print('text embeddings:', text_embeddings.shape)
print('text norm min/max:', np.linalg.norm(text_embeddings, axis=1).min(), np.linalg.norm(text_embeddings, axis=1).max())

Text metadata columns: ['class_index', 'action_label', 'action_text', 'prompt']


,class_index,action_label,action_text,prompt
0,0,SIL,SIL,a video of the action: SIL
1,1,pour_cereals,pour cereals,a video of the action: pour cereals
2,2,pour_milk,pour milk,a video of the action: pour milk
3,3,stir_cereals,stir cereals,a video of the action: stir cereals
4,4,take_bowl,take bowl,a video of the action: take bowl


Detected text label column: action_label
num_classes: 48
train/test: 1460 252
text embeddings: (48, 512)
text norm min/max: 0.99999994 1.0


In [4]:
# =====================
# Verify data and compute normalization statistics
# =====================
def load_label_ids(video_id):
    labs = read_lines(GT_DIR / f'{video_id}.txt')
    return np.array([label_to_id[x] for x in labs], dtype=np.int64)

rows = []
nan_count = 0
for vid in train_ids + test_ids:
    fp = FEATURE_DIR / f'{vid}.npy'
    gp = GT_DIR / f'{vid}.txt'
    assert fp.exists(), fp
    assert gp.exists(), gp
    x = np.load(fp)
    y = load_label_ids(vid)
    assert x.shape == (INPUT_DIM, len(y)), (vid, x.shape, len(y))
    nan_count += int(np.isnan(x).sum())
    rows.append({'video_id': vid, 'D': x.shape[0], 'T': x.shape[1], 'split': 'train' if vid in set(train_ids) else 'test'})

df_check = pd.DataFrame(rows)
print('rows:', len(df_check))
print('feature dims:', sorted(df_check['D'].unique().tolist()))
print('feature lengths:', sorted(df_check['T'].unique().tolist()))
print('total NaNs:', nan_count)
display(df_check.head())
assert nan_count == 0


def compute_train_stats(ids):
    s = np.zeros((INPUT_DIM,), dtype=np.float64)
    ss = np.zeros((INPUT_DIM,), dtype=np.float64)
    n = 0
    for vid in ids:
        x = np.load(FEATURE_DIR / f'{vid}.npy').astype(np.float32)
        s += x.sum(axis=1)
        ss += (x ** 2).sum(axis=1)
        n += x.shape[1]
    mean = s / n
    var = ss / n - mean ** 2
    std = np.sqrt(np.maximum(var, 1e-12))
    return mean.astype(np.float32), std.astype(np.float32)

if STANDARDIZE_FEATURES:
    train_mean, train_std = compute_train_stats(train_ids)
else:
    train_mean = np.zeros((INPUT_DIM,), dtype=np.float32)
    train_std = np.ones((INPUT_DIM,), dtype=np.float32)

np.save(RESULTS_DIR / 'train_feature_mean.npy', train_mean)
np.save(RESULTS_DIR / 'train_feature_std.npy', train_std)

print('standardize:', STANDARDIZE_FEATURES)
print('mean/std:', train_mean.shape, train_std.shape)
print('std min/max:', float(train_std.min()), float(train_std.max()))

rows: 1712
feature dims: [512]
feature lengths: [16]
total NaNs: 0


,video_id,D,T,split
0,P16_cam01_P16_cereals,512,16,train
1,P16_cam01_P16_friedegg,512,16,train
2,P16_cam01_P16_juice,512,16,train
3,P16_cam01_P16_milk,512,16,train
4,P16_cam01_P16_pancake,512,16,train


standardize: True
mean/std: (512,) (512,)
std min/max: 0.20662075281143188 0.7367683053016663


In [5]:
# =====================
# Dataset
# =====================
class BreakfastPrivilegedTextDataset(Dataset):
    def __init__(self, ids, feature_dir, gt_dir, label_to_id, text_embeddings, mean, std):
        self.ids = list(ids)
        self.feature_dir = Path(feature_dir)
        self.gt_dir = Path(gt_dir)
        self.label_to_id = dict(label_to_id)
        self.text_embeddings = text_embeddings.astype(np.float32)
        self.mean = mean.astype(np.float32)
        self.std = std.astype(np.float32)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        vid = self.ids[i]
        x = np.load(self.feature_dir / f'{vid}.npy').astype(np.float32)  # [512, 16]
        labs = read_lines(self.gt_dir / f'{vid}.txt')
        y = np.array([self.label_to_id[z] for z in labs], dtype=np.int64)  # [16]
        x = (x - self.mean[:, None]) / (self.std[:, None] + 1e-8)

        # Privileged ground-truth text sequence. This is used only by the teacher.
        priv_text = self.text_embeddings[y].T.astype(np.float32)  # [512, 16]

        return {
            'video_id': vid,
            'features': torch.from_numpy(x),
            'priv_text': torch.from_numpy(priv_text),
            'labels': torch.from_numpy(y),
        }

train_ds = BreakfastPrivilegedTextDataset(train_ids, FEATURE_DIR, GT_DIR, label_to_id, text_embeddings, train_mean, train_std)
test_ds = BreakfastPrivilegedTextDataset(test_ids, FEATURE_DIR, GT_DIR, label_to_id, text_embeddings, train_mean, train_std)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

batch = next(iter(train_loader))
print('features:', batch['features'].shape)
print('priv_text:', batch['priv_text'].shape)
print('labels:', batch['labels'].shape)
assert batch['features'].shape[1:] == (INPUT_DIM, 16)
assert batch['priv_text'].shape[1:] == (TEXT_DIM, 16)
assert batch['labels'].shape[1:] == (16,)

features: torch.Size([64, 512, 16])
priv_text: torch.Size([64, 512, 16])
labels: torch.Size([64, 16])


In [6]:
# =====================
# Models
# =====================
class DilatedResidualLayer(nn.Module):
    def __init__(self, dilation, channels, dropout):
        super().__init__()
        self.conv_dilated = nn.Conv1d(channels, channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.conv_1x1 = nn.Conv1d(channels, channels, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = F.relu(self.conv_dilated(x))
        out = self.conv_1x1(out)
        out = self.dropout(out)
        return x + out

class SingleStageModel(nn.Module):
    def __init__(self, num_layers, num_f_maps, input_dim, num_classes, dropout):
        super().__init__()
        self.conv_in = nn.Conv1d(input_dim, num_f_maps, kernel_size=1)
        self.layers = nn.ModuleList([DilatedResidualLayer(2 ** i, num_f_maps, dropout) for i in range(num_layers)])
        self.conv_out = nn.Conv1d(num_f_maps, num_classes, kernel_size=1)

    def forward(self, x):
        out = self.conv_in(x)
        for layer in self.layers:
            out = layer(out)
        return self.conv_out(out)

class MultiStageModel(nn.Module):
    def __init__(self, num_stages, num_layers, num_f_maps, input_dim, num_classes, dropout):
        super().__init__()
        self.stage1 = SingleStageModel(num_layers, num_f_maps, input_dim, num_classes, dropout)
        self.stages = nn.ModuleList([
            SingleStageModel(num_layers, num_f_maps, num_classes, num_classes, dropout)
            for _ in range(num_stages - 1)
        ])

    def forward(self, x):
        outputs = []
        out = self.stage1(x)
        outputs.append(out)
        for stage in self.stages:
            out = stage(F.softmax(out, dim=1))
            outputs.append(out)
        return torch.stack(outputs, dim=0)  # [S, B, C, T]

class PrivilegedTextTeacher(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = MultiStageModel(NUM_STAGES, NUM_LAYERS, NUM_F_MAPS, INPUT_DIM + TEXT_DIM, num_classes, DROPOUT)

    def forward(self, video_x, text_x):
        return self.model(torch.cat([video_x, text_x], dim=1))


def make_video_student():
    return MultiStageModel(NUM_STAGES, NUM_LAYERS, NUM_F_MAPS, INPUT_DIM, num_classes, DROPOUT).to(DEVICE)


def make_teacher():
    return PrivilegedTextTeacher().to(DEVICE)


def count_params(m):
    return sum(p.numel() for p in m.parameters())

s = make_video_student()
t = make_teacher()
with torch.no_grad():
    x = batch['features'].to(DEVICE)
    tx = batch['priv_text'].to(DEVICE)
    print('student output:', s(x).shape)
    print('teacher output:', t(x, tx).shape)
print('student params:', count_params(s))
print('teacher params:', count_params(t))
del s, t
if torch.cuda.is_available():
    torch.cuda.empty_cache()

student output: torch.Size([4, 64, 48, 16])
teacher output: torch.Size([4, 64, 48, 16])
student params: 715200
teacher params: 747968


In [7]:
# =====================
# Metrics
# =====================
def collapse_segments(frame_labels, bg_classes=None):
    bg_classes = set(bg_classes or [])
    labels, starts, ends = [], [], []
    last = None
    for i, lab in enumerate(frame_labels):
        if lab in bg_classes:
            if last is not None:
                ends.append(i)
                last = None
            continue
        if lab != last:
            if last is not None:
                ends.append(i)
            labels.append(lab)
            starts.append(i)
            last = lab
    if last is not None:
        ends.append(len(frame_labels))
    return labels, starts, ends


def levenshtein_distance(pred, gt):
    m, n = len(pred), len(gt)
    dp = np.zeros((m + 1, n + 1), dtype=np.int32)
    dp[:, 0] = np.arange(m + 1)
    dp[0, :] = np.arange(n + 1)
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if pred[i - 1] == gt[j - 1] else 1
            dp[i, j] = min(dp[i - 1, j] + 1, dp[i, j - 1] + 1, dp[i - 1, j - 1] + cost)
    return int(dp[m, n])


def edit_score_single(pred_labels, gt_labels, bg_classes=None):
    p, _, _ = collapse_segments(pred_labels, bg_classes)
    y, _, _ = collapse_segments(gt_labels, bg_classes)
    if len(p) == 0 and len(y) == 0:
        return 100.0
    if max(len(p), len(y)) == 0:
        return 0.0
    return (1.0 - levenshtein_distance(p, y) / max(len(p), len(y))) * 100.0


def f_score_single(pred_labels, gt_labels, overlap, bg_classes=None):
    p_label, p_start, p_end = collapse_segments(pred_labels, bg_classes)
    y_label, y_start, y_end = collapse_segments(gt_labels, bg_classes)
    tp, fp = 0, 0
    hits = np.zeros(len(y_label), dtype=np.float32)
    for j in range(len(p_label)):
        best_iou, best_idx = 0.0, -1
        for i in range(len(y_label)):
            if p_label[j] != y_label[i]:
                continue
            inter = min(p_end[j], y_end[i]) - max(p_start[j], y_start[i])
            union = max(p_end[j], y_end[i]) - min(p_start[j], y_start[i])
            iou = max(inter, 0) / union if union > 0 else 0.0
            if iou > best_iou:
                best_iou, best_idx = iou, i
        if best_iou >= overlap and best_idx >= 0 and hits[best_idx] == 0:
            tp += 1
            hits[best_idx] = 1
        else:
            fp += 1
    fn = len(y_label) - int(hits.sum())
    return tp, fp, fn


def compute_metrics(pred_by_video, gt_by_video):
    bg = [x for x in label_to_id if x.lower() in {'background', 'sil', 'silence'}]
    total_correct, total_frames = 0, 0
    edits = []
    fstats = {0.10: [0, 0, 0], 0.25: [0, 0, 0], 0.50: [0, 0, 0]}
    for vid, pred in pred_by_video.items():
        gt = gt_by_video[vid]
        assert len(pred) == len(gt)
        total_correct += int((pred == gt).sum())
        total_frames += len(gt)
        pred_l = [id_to_label[int(x)] for x in pred]
        gt_l = [id_to_label[int(x)] for x in gt]
        edits.append(edit_score_single(pred_l, gt_l, bg))
        for ov in fstats:
            tp, fp, fn = f_score_single(pred_l, gt_l, ov, bg)
            fstats[ov][0] += tp
            fstats[ov][1] += fp
            fstats[ov][2] += fn
    out = {'acc': 100.0 * total_correct / max(total_frames, 1), 'edit': float(np.mean(edits))}
    for ov, (tp, fp, fn) in fstats.items():
        prec = tp / max(tp + fp, 1e-10)
        rec = tp / max(tp + fn, 1e-10)
        out[f'f1@{int(100 * ov)}'] = 100.0 * (2 * prec * rec / max(prec + rec, 1e-10))
    return out


def round_metrics(m):
    return {k: round(float(v), 2) for k, v in m.items()}


def selection_score(m):
    return float(m['f1@25'] + 0.01 * m['edit'])

In [8]:
# =====================
# Losses, eval, checkpointing
# =====================
def mstcn_loss(outputs, targets):
    total = 0.0
    for s_idx in range(outputs.shape[0]):
        logits = outputs[s_idx]
        ce = F.cross_entropy(logits.permute(0, 2, 1).reshape(-1, logits.shape[1]), targets.reshape(-1))
        log_probs = F.log_softmax(logits, dim=1)
        smooth = F.mse_loss(log_probs[:, :, 1:], log_probs.detach()[:, :, :-1], reduction='none')
        smooth = torch.clamp(smooth, min=0.0, max=16.0).mean()
        total = total + ce + SMOOTHING_WEIGHT * smooth
    return total


def kd_loss(student_outputs, teacher_outputs, targets, lam):
    ce = mstcn_loss(student_outputs, targets)
    kd_total = 0.0
    for s_idx in range(student_outputs.shape[0]):
        sl = student_outputs[s_idx]
        tl = teacher_outputs[s_idx].detach()
        log_p = F.log_softmax(sl / KD_TEMPERATURE, dim=1)
        q = F.softmax(tl / KD_TEMPERATURE, dim=1)
        kd_total = kd_total + F.kl_div(log_p, q, reduction='batchmean') * (KD_TEMPERATURE ** 2)
    kd_total = kd_total / student_outputs.shape[0]
    total = ce + lam * kd_total
    return total, {'ce': float(ce.detach().cpu()), 'kd': float(kd_total.detach().cpu()), 'total': float(total.detach().cpu())}

@torch.no_grad()
def evaluate_video(model, loader, save_dir=None):
    model.eval()
    pred_by_video, gt_by_video = {}, {}
    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
    for batch in loader:
        x = batch['features'].to(DEVICE, non_blocking=True)
        y = batch['labels'].cpu().numpy()
        vids = list(batch['video_id'])
        pred = model(x)[-1].argmax(dim=1).detach().cpu().numpy()
        for i, vid in enumerate(vids):
            pred_ids = pred[i].astype(np.int64)
            gt_ids = y[i].astype(np.int64)
            pred_by_video[vid] = pred_ids
            gt_by_video[vid] = gt_ids
            if save_dir is not None:
                labs = [id_to_label[int(z)] for z in pred_ids]
                (save_dir / f'{vid}.txt').write_text('\n'.join(labs) + '\n')
    return compute_metrics(pred_by_video, gt_by_video), pred_by_video, gt_by_video

@torch.no_grad()
def evaluate_teacher(model, loader, save_dir=None):
    model.eval()
    pred_by_video, gt_by_video = {}, {}
    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
    for batch in loader:
        x = batch['features'].to(DEVICE, non_blocking=True)
        t = batch['priv_text'].to(DEVICE, non_blocking=True)
        y = batch['labels'].cpu().numpy()
        vids = list(batch['video_id'])
        pred = model(x, t)[-1].argmax(dim=1).detach().cpu().numpy()
        for i, vid in enumerate(vids):
            pred_ids = pred[i].astype(np.int64)
            gt_ids = y[i].astype(np.int64)
            pred_by_video[vid] = pred_ids
            gt_by_video[vid] = gt_ids
            if save_dir is not None:
                labs = [id_to_label[int(z)] for z in pred_ids]
                (save_dir / f'{vid}.txt').write_text('\n'.join(labs) + '\n')
    return compute_metrics(pred_by_video, gt_by_video), pred_by_video, gt_by_video


def save_ckpt(path, model, optimizer, epoch, metrics, config):
    torch.save({
        'epoch': int(epoch),
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict() if optimizer is not None else None,
        'metrics': metrics,
        'config': config,
    }, path)

In [9]:
# =====================
# Train CE-only student and privileged teacher
# =====================
def train_video_ce(name, epochs):
    set_seed(SEED)
    model = make_video_student()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_score, best_epoch, best_metrics = -1, None, None
    hist = []
    best_path = MODEL_DIR / f'{name}_best.pt'
    last_path = MODEL_DIR / f'{name}_last.pt'
    start = time.time()
    for ep in range(1, epochs + 1):
        model.train(); losses = []
        for batch in train_loader:
            x = batch['features'].to(DEVICE, non_blocking=True)
            y = batch['labels'].to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = mstcn_loss(model(x), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step(); losses.append(float(loss.detach().cpu()))
        sch.step()
        row = {'model': name, 'epoch': ep, 'train_loss': float(np.mean(losses)), 'lr': float(sch.get_last_lr()[0])}
        if ep == 1 or ep % EVAL_EVERY == 0 or ep == epochs:
            metrics, _, _ = evaluate_video(model, test_loader)
            row.update(metrics)
            score = selection_score(metrics)
            if score > best_score:
                best_score, best_epoch, best_metrics = score, ep, dict(metrics)
                save_ckpt(best_path, model, opt, ep, metrics, {'model': name})
            print(f'[{name}] ep {ep:03d} loss={row["train_loss"]:.4f} acc={metrics["acc"]:.2f} edit={metrics["edit"]:.2f} f1@25={metrics["f1@25"]:.2f} f1@50={metrics["f1@50"]:.2f}')
        else:
            print(f'[{name}] ep {ep:03d} loss={row["train_loss"]:.4f}')
        hist.append(row)
    save_ckpt(last_path, model, opt, epochs, {}, {'model': name})
    hist_df = pd.DataFrame(hist)
    hist_path = RESULTS_DIR / f'{name}_history.csv'
    hist_df.to_csv(hist_path, index=False)
    print(name, 'elapsed min:', (time.time() - start) / 60)
    print(name, 'best_epoch:', best_epoch, 'best_metrics:', round_metrics(best_metrics))
    return {'name': name, 'model': model, 'best_path': best_path, 'last_path': last_path, 'history_path': hist_path, 'best_epoch': best_epoch, 'best_metrics': best_metrics}


def train_privileged_teacher():
    set_seed(SEED + 1)
    model = make_teacher()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=TEACHER_EPOCHS)
    best_score, best_epoch, best_metrics = -1, None, None
    hist = []
    best_path = MODEL_DIR / 'privileged_teacher_best.pt'
    last_path = MODEL_DIR / 'privileged_teacher_last.pt'
    start = time.time()
    for ep in range(1, TEACHER_EPOCHS + 1):
        model.train(); losses = []
        for batch in train_loader:
            x = batch['features'].to(DEVICE, non_blocking=True)
            t = batch['priv_text'].to(DEVICE, non_blocking=True)
            y = batch['labels'].to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = mstcn_loss(model(x, t), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step(); losses.append(float(loss.detach().cpu()))
        sch.step()
        row = {'model': 'privileged_teacher', 'epoch': ep, 'train_loss': float(np.mean(losses)), 'lr': float(sch.get_last_lr()[0])}
        if ep == 1 or ep % EVAL_EVERY == 0 or ep == TEACHER_EPOCHS:
            metrics, _, _ = evaluate_teacher(model, test_loader)
            row.update(metrics)
            score = selection_score(metrics)
            if score > best_score:
                best_score, best_epoch, best_metrics = score, ep, dict(metrics)
                save_ckpt(best_path, model, opt, ep, metrics, {'model': 'privileged_teacher', 'oracle': True})
            print(f'[teacher] ep {ep:03d} loss={row["train_loss"]:.4f} acc={metrics["acc"]:.2f} edit={metrics["edit"]:.2f} f1@25={metrics["f1@25"]:.2f} f1@50={metrics["f1@50"]:.2f}')
        else:
            print(f'[teacher] ep {ep:03d} loss={row["train_loss"]:.4f}')
        hist.append(row)
    save_ckpt(last_path, model, opt, TEACHER_EPOCHS, {}, {'model': 'privileged_teacher'})
    hist_df = pd.DataFrame(hist)
    hist_path = RESULTS_DIR / 'privileged_teacher_history.csv'
    hist_df.to_csv(hist_path, index=False)
    print('teacher elapsed min:', (time.time() - start) / 60)
    print('teacher best_epoch:', best_epoch, 'best_metrics:', round_metrics(best_metrics))
    return {'model': model, 'best_path': best_path, 'last_path': last_path, 'history_path': hist_path, 'best_epoch': best_epoch, 'best_metrics': best_metrics}

ce_result = train_video_ce('controlled_ce_student', CE_EPOCHS)
teacher_result = train_privileged_teacher()

[controlled_ce_student] ep 001 loss=14.2955 acc=17.93 edit=11.66 f1@25=9.89 f1@50=4.26
[controlled_ce_student] ep 002 loss=12.7599
[controlled_ce_student] ep 003 loss=11.8696
[controlled_ce_student] ep 004 loss=10.9521
[controlled_ce_student] ep 005 loss=10.2864 acc=19.20 edit=11.89 f1@25=11.01 f1@50=5.79
[controlled_ce_student] ep 006 loss=9.6456
[controlled_ce_student] ep 007 loss=9.0713
[controlled_ce_student] ep 008 loss=8.6584
[controlled_ce_student] ep 009 loss=8.1774
[controlled_ce_student] ep 010 loss=7.7076 acc=37.08 edit=29.19 f1@25=27.87 f1@50=15.11
[controlled_ce_student] ep 011 loss=7.3702
[controlled_ce_student] ep 012 loss=7.1175
[controlled_ce_student] ep 013 loss=6.8283
[controlled_ce_student] ep 014 loss=6.6284
[controlled_ce_student] ep 015 loss=6.5165 acc=48.31 edit=46.07 f1@25=46.63 f1@50=28.83
[controlled_ce_student] ep 016 loss=6.4260
[controlled_ce_student] ep 017 loss=6.2683
[controlled_ce_student] ep 018 loss=6.0928
[controlled_ce_student] ep 019 loss=6.0351
[

In [10]:
# =====================
# Reload best teacher and save oracle predictions
# =====================
teacher = make_teacher()
teacher_ckpt = torch.load(teacher_result['best_path'], map_location=DEVICE)
teacher.load_state_dict(teacher_ckpt['model_state_dict'])
teacher.eval()

teacher_pred_dir = PRED_DIR / 'privileged_teacher_best_predictions'
teacher_metrics, _, _ = evaluate_teacher(teacher, test_loader, save_dir=teacher_pred_dir)
teacher_metrics_rounded = round_metrics(teacher_metrics)

print('teacher best epoch:', teacher_ckpt['epoch'])
print(json.dumps(teacher_metrics_rounded, indent=2))
print('prediction files:', len(list(teacher_pred_dir.glob('*.txt'))))
assert len(list(teacher_pred_dir.glob('*.txt'))) == len(test_ids)

teacher best epoch: 65
{
  "acc": 98.26,
  "edit": 96.26,
  "f1@10": 96.9,
  "f1@25": 96.9,
  "f1@50": 96.71
}
prediction files: 252


In [11]:
# =====================
# Train KD students with small lambda sweep
# =====================
def train_kd_student(lam):
    set_seed(SEED + int(lam * 1000) + 10)
    student = make_video_student()
    opt = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=STUDENT_EPOCHS)
    safe_lam = str(lam).replace('.', 'p')
    best_path = MODEL_DIR / f'kd_student_lambda_{safe_lam}_best.pt'
    last_path = MODEL_DIR / f'kd_student_lambda_{safe_lam}_last.pt'
    hist_path = RESULTS_DIR / f'kd_student_lambda_{safe_lam}_history.csv'
    best_score, best_epoch, best_metrics = -1, None, None
    hist = []
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad_(False)
    start = time.time()
    for ep in range(1, STUDENT_EPOCHS + 1):
        student.train(); losses = []; ces = []; kds = []
        for batch in train_loader:
            x = batch['features'].to(DEVICE, non_blocking=True)
            t = batch['priv_text'].to(DEVICE, non_blocking=True)
            y = batch['labels'].to(DEVICE, non_blocking=True)
            with torch.no_grad():
                teacher_outputs = teacher(x, t)
            opt.zero_grad(set_to_none=True)
            student_outputs = student(x)
            loss, parts = kd_loss(student_outputs, teacher_outputs, y, lam=lam)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), GRAD_CLIP)
            opt.step()
            losses.append(parts['total']); ces.append(parts['ce']); kds.append(parts['kd'])
        sch.step()
        row = {'model': 'kd_student', 'kd_lambda': lam, 'epoch': ep, 'train_loss': float(np.mean(losses)), 'train_ce': float(np.mean(ces)), 'train_kd': float(np.mean(kds)), 'lr': float(sch.get_last_lr()[0])}
        if ep == 1 or ep % EVAL_EVERY == 0 or ep == STUDENT_EPOCHS:
            metrics, _, _ = evaluate_video(student, test_loader)
            row.update(metrics)
            score = selection_score(metrics)
            if score > best_score:
                best_score, best_epoch, best_metrics = score, ep, dict(metrics)
                save_ckpt(best_path, student, opt, ep, metrics, {'model': 'kd_student', 'kd_lambda': lam, 'temperature': KD_TEMPERATURE, 'teacher': str(teacher_result['best_path'])})
            print(f'[KD {lam}] ep {ep:03d} loss={row["train_loss"]:.4f} ce={row["train_ce"]:.4f} kd={row["train_kd"]:.4f} acc={metrics["acc"]:.2f} edit={metrics["edit"]:.2f} f1@25={metrics["f1@25"]:.2f} f1@50={metrics["f1@50"]:.2f}')
        else:
            print(f'[KD {lam}] ep {ep:03d} loss={row["train_loss"]:.4f} ce={row["train_ce"]:.4f} kd={row["train_kd"]:.4f}')
        hist.append(row)
    save_ckpt(last_path, student, opt, STUDENT_EPOCHS, {}, {'model': 'kd_student', 'kd_lambda': lam})
    hist_df = pd.DataFrame(hist)
    hist_df.to_csv(hist_path, index=False)
    print('KD', lam, 'elapsed min:', (time.time() - start) / 60)
    print('KD', lam, 'best_epoch:', best_epoch, 'best_metrics:', round_metrics(best_metrics))
    return {'kd_lambda': lam, 'model': student, 'best_path': best_path, 'last_path': last_path, 'history_path': hist_path, 'best_epoch': best_epoch, 'best_metrics': best_metrics}

kd_results = []
for lam in KD_LAMBDAS:
    print('\n' + '=' * 80)
    print('Training KD lambda =', lam)
    print('=' * 80)
    kd_results.append(train_kd_student(lam))

kd_summary = []
for r in kd_results:
    row = {'kd_lambda': r['kd_lambda'], 'best_epoch': r['best_epoch']}
    row.update(round_metrics(r['best_metrics']))
    kd_summary.append(row)
kd_summary_df = pd.DataFrame(kd_summary)
display(kd_summary_df)
kd_summary_path = RESULTS_DIR / 'kd_lambda_sweep_summary.csv'
kd_summary_df.to_csv(kd_summary_path, index=False)
print('saved:', kd_summary_path)


Training KD lambda = 0.01
[KD 0.01] ep 001 loss=14.8521 ce=14.2453 kd=60.6761 acc=11.68 edit=0.00 f1@25=0.00 f1@50=0.00
[KD 0.01] ep 002 loss=13.2910 ce=12.7314 kd=55.9558
[KD 0.01] ep 003 loss=12.4402 ce=11.9080 kd=53.2196
[KD 0.01] ep 004 loss=11.4886 ce=10.9877 kd=50.0889
[KD 0.01] ep 005 loss=10.8083 ce=10.3299 kd=47.8399 acc=18.33 edit=11.39 f1@25=10.61 f1@50=6.17
[KD 0.01] ep 006 loss=10.1775 ce=9.7134 kd=46.4086
[KD 0.01] ep 007 loss=9.6222 ce=9.1771 kd=44.5075
[KD 0.01] ep 008 loss=8.9969 ce=8.5680 kd=42.8909
[KD 0.01] ep 009 loss=8.4336 ce=8.0254 kd=40.8261
[KD 0.01] ep 010 loss=8.0068 ce=7.6164 kd=39.0380 acc=40.53 edit=31.91 f1@25=34.04 f1@50=19.49
[KD 0.01] ep 011 loss=7.6053 ce=7.2288 kd=37.6420
[KD 0.01] ep 012 loss=7.2486 ce=6.8858 kd=36.2866
[KD 0.01] ep 013 loss=7.0057 ce=6.6549 kd=35.0759
[KD 0.01] ep 014 loss=6.8128 ce=6.4728 kd=34.0013
[KD 0.01] ep 015 loss=6.7033 ce=6.3648 kd=33.8538 acc=48.14 edit=45.38 f1@25=46.09 f1@50=30.55
[KD 0.01] ep 016 loss=6.5419 ce=6.21

,kd_lambda,best_epoch,acc,edit,f1@10,f1@25,f1@50
0,0.01,60,56.77,54.62,55.18,53.93,45.18
1,0.05,95,57.99,56.46,56.67,55.42,46.19
2,0.10,50,56.30,55.05,56.09,54.18,42.68


saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_privileged_text_teacher_kd_full_split1_split1/results/kd_lambda_sweep_summary.csv


In [12]:
# =====================
# Select best KD student and save predictions
# =====================
best_kd_result = max(kd_results, key=lambda r: selection_score(r['best_metrics']))
best_kd_lambda = best_kd_result['kd_lambda']
print('best KD lambda:', best_kd_lambda)
print('best KD epoch:', best_kd_result['best_epoch'])
print('best KD metrics:', round_metrics(best_kd_result['best_metrics']))

best_kd = make_video_student()
kd_ckpt = torch.load(best_kd_result['best_path'], map_location=DEVICE)
best_kd.load_state_dict(kd_ckpt['model_state_dict'])
best_kd.eval()

safe_lam = str(best_kd_lambda).replace('.', 'p')
kd_pred_dir = PRED_DIR / f'best_kd_lambda_{safe_lam}_predictions'
kd_metrics, _, _ = evaluate_video(best_kd, test_loader, save_dir=kd_pred_dir)
kd_metrics_rounded = round_metrics(kd_metrics)

print('KD checkpoint epoch:', kd_ckpt['epoch'])
print(json.dumps(kd_metrics_rounded, indent=2))
print('prediction files:', len(list(kd_pred_dir.glob('*.txt'))))
assert len(list(kd_pred_dir.glob('*.txt'))) == len(test_ids)

best KD lambda: 0.05
best KD epoch: 95
best KD metrics: {'acc': 57.99, 'edit': 56.46, 'f1@10': 56.67, 'f1@25': 55.42, 'f1@50': 46.19}
KD checkpoint epoch: 95
{
  "acc": 57.99,
  "edit": 56.46,
  "f1@10": 56.67,
  "f1@25": 55.42,
  "f1@50": 46.19
}
prediction files: 252


In [13]:
# =====================
# Final tables and summary
# =====================
ce_metrics_rounded = round_metrics(ce_result['best_metrics'])

final_rows = [
    {
        'experiment': 'previous_hidden_visual_only_baseline',
        'features': 'ProcedureVRL hidden video embeddings',
        'feature_shape': '[512, 16]',
        'model': 'MS-TCN-style visual-only',
        'inference_input': 'video only',
        'best_epoch': PREVIOUS_HIDDEN_BASELINE['best_epoch'],
        'acc': PREVIOUS_HIDDEN_BASELINE['acc'],
        'edit': PREVIOUS_HIDDEN_BASELINE['edit'],
        'f1@10': PREVIOUS_HIDDEN_BASELINE['f1@10'],
        'f1@25': PREVIOUS_HIDDEN_BASELINE['f1@25'],
        'f1@50': PREVIOUS_HIDDEN_BASELINE['f1@50'],
        'note': PREVIOUS_HIDDEN_BASELINE['note'],
    },
    {
        'experiment': 'controlled_ce_student',
        'features': 'ProcedureVRL hidden video embeddings',
        'feature_shape': '[512, 16]',
        'model': 'CE-only video student',
        'inference_input': 'video only',
        'best_epoch': int(ce_result['best_epoch']),
        **ce_metrics_rounded,
        'note': 'Controlled same-notebook CE-only student.',
    },
    {
        'experiment': 'privileged_text_teacher_oracle',
        'features': 'ProcedureVRL hidden + ground-truth text sequence',
        'feature_shape': '[512, 16] + [512, 16]',
        'model': 'Privileged text teacher',
        'inference_input': 'video + ground-truth text',
        'best_epoch': int(teacher_result['best_epoch']),
        **teacher_metrics_rounded,
        'note': 'Oracle teacher; not deployable at inference.',
    },
    {
        'experiment': 'privileged_text_kd_student',
        'features': 'ProcedureVRL hidden video embeddings',
        'feature_shape': '[512, 16]',
        'model': 'Video-only KD student',
        'inference_input': 'video only',
        'best_epoch': int(best_kd_result['best_epoch']),
        **kd_metrics_rounded,
        'note': f'Best KD student from lambda sweep; lambda={best_kd_lambda}, T={KD_TEMPERATURE}.',
    },
]

final_df = pd.DataFrame(final_rows)
final_csv = RESULTS_DIR / 'final_metrics.csv'
final_md = RESULTS_DIR / 'final_metrics.md'
final_df.to_csv(final_csv, index=False)
final_md.write_text(final_df.to_markdown(index=False))
display(final_df)
print('saved:', final_csv)
print('saved:', final_md)

comparison_rows = [
    {'experiment': 'I3D baseline', 'features': 'I3D', 'model': 'Official MS-TCN visual-only, 30 epochs', 'feature_shape': '[2048, T]', 'acc': 55.38, 'edit': 44.97, 'f1@10': 39.05, 'f1@25': 34.80, 'f1@50': 25.25, 'note': 'Previous frame-level baseline.'},
    {'experiment': 'I3D + CLIP KD proof-of-concept', 'features': 'I3D + CLIP text teacher', 'model': 'Video-only KD student, lambda=0.05', 'feature_shape': '[2048, T]', 'acc': 70.98, 'edit': 58.53, 'f1@10': 50.16, 'f1@25': 46.86, 'f1@50': 38.33, 'note': 'Useful proof-of-concept; visual/text spaces not aligned.'},
    {'experiment': 'ProcedureVRL coarse baseline', 'features': 'ProcedureVRL coarse outputs', 'model': 'MS-TCN-style visual-only', 'feature_shape': '[9871, 16]', 'acc': 48.91, 'edit': 49.15, 'f1@10': 55.46, 'f1@25': 54.38, 'f1@50': 42.61, 'note': 'Coarse output-level features.'},
    {'experiment': 'ProcedureVRL hidden baseline', 'features': 'ProcedureVRL hidden embeddings', 'model': 'MS-TCN-style visual-only', 'feature_shape': '[512, 16]', 'acc': PREVIOUS_HIDDEN_BASELINE['acc'], 'edit': PREVIOUS_HIDDEN_BASELINE['edit'], 'f1@10': PREVIOUS_HIDDEN_BASELINE['f1@10'], 'f1@25': PREVIOUS_HIDDEN_BASELINE['f1@25'], 'f1@50': PREVIOUS_HIDDEN_BASELINE['f1@50'], 'note': 'Notebook 10 baseline.'},
    {'experiment': 'Controlled CE-only student', 'features': 'ProcedureVRL hidden embeddings', 'model': 'CE-only video student', 'feature_shape': '[512, 16]', **ce_metrics_rounded, 'note': 'Same-notebook controlled baseline.'},
    {'experiment': 'Privileged text teacher', 'features': 'ProcedureVRL hidden + GT text', 'model': 'Oracle teacher', 'feature_shape': '[512, 16] + [512, 16]', **teacher_metrics_rounded, 'note': 'Teacher only; not deployable.'},
    {'experiment': 'Privileged text KD student', 'features': 'ProcedureVRL hidden embeddings', 'model': 'Video-only KD student', 'feature_shape': '[512, 16]', **kd_metrics_rounded, 'note': f'Main improved text-assisted result; lambda={best_kd_lambda}.'},
]
comparison_df = pd.DataFrame(comparison_rows)
comparison_csv = RESULTS_DIR / 'comparison_with_previous_runs.csv'
comparison_md = RESULTS_DIR / 'comparison_with_previous_runs.md'
comparison_df.to_csv(comparison_csv, index=False)
comparison_md.write_text(comparison_df.to_markdown(index=False))
display(comparison_df)
print('saved:', comparison_csv)
print('saved:', comparison_md)

print('\nKD minus controlled CE-only:')
for key in ['acc', 'edit', 'f1@10', 'f1@25', 'f1@50']:
    print(f'{key}: {kd_metrics_rounded[key] - ce_metrics_rounded[key]:+.2f}')

print('\nKD minus previous notebook-10 hidden baseline:')
for key in ['acc', 'edit', 'f1@10', 'f1@25', 'f1@50']:
    print(f'{key}: {kd_metrics_rounded[key] - PREVIOUS_HIDDEN_BASELINE[key]:+.2f}')

summary = {
    'status': 'completed',
    'experiment': 'procedurevrl_hidden_privileged_text_teacher_kd_breakfast_split1',
    'run_mode': RUN_MODE,
    'data_root': str(DATA_ROOT),
    'output_root': str(OUT_ROOT),
    'num_train_videos': len(train_ids),
    'num_test_videos': len(test_ids),
    'feature_shape_per_video': [INPUT_DIM, 16],
    'privileged_text_shape_per_video': [TEXT_DIM, 16],
    'previous_hidden_baseline': PREVIOUS_HIDDEN_BASELINE,
    'controlled_ce_student': {'best_epoch': int(ce_result['best_epoch']), 'best_metrics': ce_metrics_rounded, 'checkpoint': str(ce_result['best_path'])},
    'privileged_text_teacher': {'best_epoch': int(teacher_result['best_epoch']), 'best_metrics': teacher_metrics_rounded, 'checkpoint': str(teacher_result['best_path']), 'oracle_teacher': True},
    'best_kd_student': {'best_epoch': int(best_kd_result['best_epoch']), 'best_metrics': kd_metrics_rounded, 'checkpoint': str(best_kd_result['best_path']), 'kd_lambda': float(best_kd_lambda), 'temperature': KD_TEMPERATURE},
    'paths': {'final_metrics_csv': str(final_csv), 'comparison_csv': str(comparison_csv), 'kd_lambda_sweep_summary': str(kd_summary_path), 'teacher_predictions': str(teacher_pred_dir), 'kd_predictions': str(kd_pred_dir)},
    'interpretation_note': 'The teacher uses ground-truth text and is not deployable. The final KD student is video-only at inference. The primary controlled comparison is KD student vs CE-only student in the same notebook.',
}
summary_path = RESULTS_DIR / 'final_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('\nFinal summary:')
print(json.dumps(summary, indent=2))
print('saved:', summary_path)

,experiment,features,feature_shape,model,inference_input,best_epoch,acc,edit,f1@10,f1@25,f1@50,note
0,previous_hidden_visual_only_baseline,ProcedureVRL hidden video embeddings,"[512, 16]",MS-TCN-style visual-only,video only,55,58.04,55.57,58.58,57.73,45.76,"Notebook 10, 120 epochs, best checkpoint at ep..."
1,controlled_ce_student,ProcedureVRL hidden video embeddings,"[512, 16]",CE-only video student,video only,85,57.42,55.61,55.76,54.29,45.34,Controlled same-notebook CE-only student.
2,privileged_text_teacher_oracle,ProcedureVRL hidden + ground-truth text sequence,"[512, 16] + [512, 16]",Privileged text teacher,video + ground-truth text,65,98.26,96.26,96.90,96.90,96.71,Oracle teacher; not deployable at inference.
3,privileged_text_kd_student,ProcedureVRL hidden video embeddings,"[512, 16]",Video-only KD student,video only,95,57.99,56.46,56.67,55.42,46.19,Best KD student from lambda sweep; lambda=0.05...


saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_privileged_text_teacher_kd_full_split1_split1/results/final_metrics.csv
saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_privileged_text_teacher_kd_full_split1_split1/results/final_metrics.md


,experiment,features,model,feature_shape,acc,edit,f1@10,f1@25,f1@50,note
0,I3D baseline,I3D,"Official MS-TCN visual-only, 30 epochs","[2048, T]",55.38,44.97,39.05,34.80,25.25,Previous frame-level baseline.
1,I3D + CLIP KD proof-of-concept,I3D + CLIP text teacher,"Video-only KD student, lambda=0.05","[2048, T]",70.98,58.53,50.16,46.86,38.33,Useful proof-of-concept; visual/text spaces no...
2,ProcedureVRL coarse baseline,ProcedureVRL coarse outputs,MS-TCN-style visual-only,"[9871, 16]",48.91,49.15,55.46,54.38,42.61,Coarse output-level features.
3,ProcedureVRL hidden baseline,ProcedureVRL hidden embeddings,MS-TCN-style visual-only,"[512, 16]",58.04,55.57,58.58,57.73,45.76,Notebook 10 baseline.
4,Controlled CE-only student,ProcedureVRL hidden embeddings,CE-only video student,"[512, 16]",57.42,55.61,55.76,54.29,45.34,Same-notebook controlled baseline.
5,Privileged text teacher,ProcedureVRL hidden + GT text,Oracle teacher,"[512, 16] + [512, 16]",98.26,96.26,96.90,96.90,96.71,Teacher only; not deployable.
6,Privileged text KD student,ProcedureVRL hidden embeddings,Video-only KD student,"[512, 16]",57.99,56.46,56.67,55.42,46.19,Main improved text-assisted result; lambda=0.05.


saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_privileged_text_teacher_kd_full_split1_split1/results/comparison_with_previous_runs.csv
saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_privileged_text_teacher_kd_full_split1_split1/results/comparison_with_previous_runs.md

KD minus controlled CE-only:
acc: +0.57
edit: +0.85
f1@10: +0.91
f1@25: +1.13
f1@50: +0.85

KD minus previous notebook-10 hidden baseline:
acc: -0.05
edit: +0.89
f1@10: -1.91
f1@25: -2.31
f1@50: +0.43

Final summary:
{
  "status": "completed",
  "experiment": "procedurevrl_hidden_privileged_text_teacher_kd_breakfast_split1",
  "run_mode": "full_split1",
  "data_root": "/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_full_split1_split1_views16/mstcn_format",
  "output_root": "/content/drive/MyDrive/mmf_tas_lab_data